## Data Inspection

The dataset contains historical stock price data for S&P 500 companies from 2000 onwards.

Initial inspection showed:

- 2,931,968 observations
- 7 variables
- Stock prices including open, high, low and close values
- Daily trading volume
- Company ticker symbols

This project investigates whether historical stock performance can help identify future outperformers and whether lower volatility stocks generate superior risk-adjusted returns.

In [2]:
import os

os.getcwd()

'c:\\Users\\Owner\\Desktop\\sql s&p analysis\\Notebooks'

In [3]:
import os

os.chdir(r"C:\Users\Owner\Desktop\sql s&p analysis")

os.getcwd()

'C:\\Users\\Owner\\Desktop\\sql s&p analysis'

In [4]:
import pandas as pd

stocks = pd.read_csv("sp500_stocks.csv")

stocks.head()

FileNotFoundError: [Errno 2] No such file or directory: 'sp500_stocks.csv'

In [ ]:
stocks.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2931968 entries, 0 to 2931967
Data columns (total 7 columns):
 #   Column  Dtype  
---  ------  -----  
 0   date    object 
 1   open    float64
 2   high    float64
 3   low     float64
 4   close   float64
 5   volume  float64
 6   symbol  object 
dtypes: float64(5), object(2)
memory usage: 156.6+ MB


## Data Quality Analysis

A data quality assessment was conducted using isnull().sum(). No missing values were found across any of the seven variables, meaning no imputation or removal of incomplete records was required before analysis.

In [ ]:
stocks.isnull().sum()

date      0
open      0
high      0
low       0
close     0
volume    0
symbol    0
dtype: int64

## Company Coverage Analysis

Before selecting companies for analysis, the number of unique stocks represented in the dataset was assessed.

The dataset contains 563 unique ticker symbols, providing broad coverage across multiple sectors of the S&P 500 over the period studied.

In [ ]:
stocks["symbol"].nunique()

503

## Sample Selection

To test momentum and risk-adjusted return hypotheses efficiently, a representative sample of large-cap companies was selected across multiple sectors.

The sample includes technology, financial, healthcare, energy and consumer-focused companies to ensure sector diversity while maintaining a manageable dataset size.

The original dataset contained 563 unique ticker symbols and approximately 2.9 million observations.

To facilitate focused hypothesis testing, a representative sample of 30 large-cap companies was selected across technology, financials, healthcare, energy and consumer sectors.

The filtering process reduced the dataset from approximately 2.9 million observations to 187,343 observations while maintaining sector and investment-style diversity.


In [ ]:
selected_stocks = [
    # Technology
    "AAPL", "MSFT", "NVDA", "META",
    "GOOGL", "ADBE", "CRM", "ORCL",

    # Financials
    "JPM", "BAC", "GS", "MS",
    "V", "MA",

    # Healthcare
    "JNJ", "PFE", "UNH",
    "ABBV", "MRK",

    # Energy
    "XOM", "CVX", "COP",

    # Consumer
    "AMZN", "WMT", "COST",
    "MCD", "NKE",

    # Industrials
    "CAT", "GE", "RTX"
]

In [ ]:
stocks_filtered = stocks[
    stocks["symbol"].isin(selected_stocks)
]

stocks_filtered.shape

(187343, 7)

In [ ]:
stocks_filtered["symbol"].nunique()

30

In [ ]:
stocks_filtered.to_csv(
    "sp500_stocks_selected.csv",
    index=False
)

In [ ]:
import os

os.path.exists("sp500_stocks_selected.csv")

True

## SQL Validation 1 - Dataset Size

The first SQL validation confirmed that the filtered dataset loaded into SQLite correctly.

Results:

- Total observations: 187,343
- Total companies: 30

This confirms that the database contains the expected number of companies selected for analysis.

## SQL Validation 2 - Missing Values

The second SQL validation checked each important column for missing values.

Results:

- Total rows: 187,343
- Date values: 187,343
- Open prices: 187,343
- High prices: 187,343
- Low prices: 187,343
- Close prices: 187,343
- Volume values: 187,343
- Company symbols: 187,343

Every column contains the expected number of observations.

This confirms there are no missing values within the filtered stock price dataset.

## SQL Validation 3 - Duplicate Records

The third SQL validation checked whether any company had more than one record for the same trading day.

Result:

- No duplicate records were found.

This confirms that each company has a single observation for each trading date, ensuring the dataset is suitable for time-series analysis.

## SQL Validation 4 - Date Range

The fourth SQL validation checked the historical coverage of the filtered stock price database.

Results:

- Earliest trading date: 2000-01-03
- Latest trading date: 2026-06-22

The dataset contains more than 26 years of historical stock price data, providing sufficient history for long-term trend, momentum and investment analysis.

## SQL Validation 5 - Records per Company

The fifth SQL validation counted the number of observations for each company included in the filtered dataset.

Results:

- 30 companies were successfully imported.
- Every company contains historical trading data.
- Companies have different numbers of observations because some entered the S&P 500 after 2000 or were listed later.

This confirms that every selected company is available for analysis and that differences in record counts are expected rather than data quality issues.

## SQL Validation 6 - Company Date Coverage

The sixth SQL validation examined the historical trading period available for each company.

Results:

- 24 companies contain complete historical data from 03/01/2000 to 22/06/2026.
- Six companies (CRM, GOOGL, MA, V, META and ABBV) begin later because they entered the public markets or the S&P 500 after 2000.
- All companies share the same latest trading date (22/06/2026).

This confirms that differences in observation counts are expected and reflect company listing dates rather than missing or incomplete data.

## SQL Validation 7 – Average Closing Price by Company

This query calculated the average historical closing price for each company across the entire dataset.

### Results

- 30 companies were analysed.
- META recorded the highest average closing price at **$232.85**.
- COST ranked second with an average closing price of **$204.78**.
- GS ranked third at **$187.47**.
- NVDA recorded the lowest average closing price at **$17.21**.
- Average prices varied considerably across companies, demonstrating substantial differences in historical valuation levels.

### Interpretation

Average share price alone does **not** indicate whether a company is a better investment because stock prices are affected by factors such as stock splits, shares outstanding and company-specific history.

However, this query provides a useful high-level comparison of historical pricing across the selected S&P 500 companies and confirms that meaningful variation exists within the dataset. These differences will be explored further using return, volatility and momentum analysis later in the project.

## SQL Analysis 8 – Average Daily Trading Volume

This query calculated the average daily trading volume for each company across the available trading history.

### Results

- 30 companies were analysed.
- NVDA recorded the highest average daily trading volume at approximately **591 million shares per day**.
- AAPL ranked second with approximately **368 million shares**, followed by **AMZN** (114 million) and **GOOGL** (110 million).
- COST recorded the lowest average daily trading volume at approximately **3.2 million shares per day**.
- Trading activity differed substantially between companies.

### Interpretation

Trading volume measures market liquidity and investor participation.

Companies with higher average trading volumes are generally easier to buy and sell because there are more active market participants. The results show that technology companies such as NVDA and AAPL attract significantly higher trading activity than many other companies in the sample.

These differences in liquidity will provide useful context when interpreting stock performance and momentum throughout the remainder of the project.

## SQL Analysis 9 – Historical Trading Range

This query identified the highest and lowest historical closing price recorded for each company in the dataset.

### Results

- 30 companies were analysed.
- GS recorded the highest historical closing price at **$1,106.37**.
- COST ranked second at **$1,094.32**, followed by **CAT** at **$1,022.28**.
- Several companies, including **AAPL**, **AMZN** and **NVDA**, recorded extremely low historical minimum prices because the dataset is adjusted for stock splits and covers more than two decades of trading history.

### Interpretation

Historical trading ranges illustrate how much a company's share price has changed over time. However, comparing absolute share prices does **not** indicate which company delivered the best investment performance because stock prices are influenced by stock splits, the number of shares outstanding and company-specific capital structures.

For this reason, the next stage of the analysis will focus on **percentage returns**, which provide a fairer comparison of long-term investment performance across companies.

## SQL Analysis 10 – Total Investment Return

This query calculated the total percentage return for each company using its first available closing price and its most recent closing price.

### Results

- 30 companies were analysed.
- NVDA produced the highest total return at **233,534.18%**.
- AAPL ranked second with a total return of **35,354.39%**.
- GOOGL ranked third with **13,949.35%**.
- PFE produced the lowest return at **122.48%**.

### Interpretation

Unlike average or maximum share prices, percentage return provides a fair comparison of investment performance across companies.

The results reveal significant differences in long-term shareholder returns. Technology companies such as NVDA, AAPL and GOOGL substantially outperformed many companies from other sectors over the sample period.

These findings provide the first evidence that certain companies consistently generated exceptional long-term returns. Further analysis of volatility and momentum will determine whether these high returns were accompanied by higher levels of investment risk.

## SQL Analysis 11 – Annualized Return (CAGR)

This query calculated the compound annual growth rate (CAGR) for each company using its first available closing price and latest closing price.

### Results

- 30 companies were analysed.
- NVDA produced the highest CAGR at **34.05%**.
- MA ranked second with a CAGR of **26.72%**.
- GOOGL ranked third at **25.41%**.
- AAPL ranked fourth at **24.83%**.
- PFE recorded the lowest CAGR at **3.07%**.

### Interpretation

CAGR is useful because it converts each company's total return into an average annual growth rate. This makes it easier to compare companies with different starting dates.

The results show that NVDA did not only produce the highest total return, but also the strongest annualized return. This strengthens the evidence that NVDA was the dominant long-term performer in the selected sample.

The next stage should compare these returns against volatility to understand whether the highest-returning companies also carried higher investment risk.

## SQL Analysis 12 – Historical Price Range

This query calculated the difference between the highest and lowest historical closing price for each company.

### Results

- 30 companies were analysed.
- COST recorded the largest historical price range at **$1,076.17**.
- GS ranked second with **$1,067.97**.
- CAT ranked third with **$1,014.33**.
- Companies displayed substantial differences in their historical trading ranges.

### Interpretation

Historical price range provides a simple measure of how much a company's share price has fluctuated over the available trading period.

However, price range should not be confused with statistical volatility because it only considers the highest and lowest observed prices rather than the variability of daily returns.

The next stage of the analysis will examine risk using more appropriate financial measures.